# 🔄 The Comparative Dashboard

**Template Type:** Comparative Analysis
**Version:** 1.0.0
**Created:** October 21, 2025
**Part of:** 7→3+1 Template Consolidation (Phase 3)

---

## Purpose

This dashboard enables **systematic multi-system comparison** by importing and analyzing synthesis journals from Phase 2. It replaces the manual entry approach of Template 04 with automated data integration.

## Key Features

✅ **Auto-Import** - Load 2-10 synthesis JSON files directly
✅ **Score Matrix** - Visual comparison across 8 performance criteria
✅ **Statistical Analysis** - Means, std devs, rankings, winners
✅ **Comparative Visualizations** - Radar charts, heatmaps, bar charts
✅ **Criterion Analysis** - Detailed qualitative comparison per criterion
✅ **Pattern Identification** - Universal strengths/limitations discovery
✅ **Use Case Mapping** - Task-based and user profile recommendations
✅ **Export** - JSON, Markdown, CSV formats

## When to Use

- After completing **2+ synthesis journals** (Template: The Synthesis Journal)
- Mid-evaluation to identify patterns across systems
- End of evaluation for final synthesis and recommendations
- When making system selection decisions

## Prerequisites

- 2-10 completed synthesis journals (`synthesis_*.json` files)
- Each synthesis should ideally include:
  - All 8 criteria scored
  - Evidence from multiple sessions
  - Journal entries documenting experience

## Expected Time

**60-90 minutes** for thorough comparative analysis

---

**Let's begin! Run cells in order (Shift+Enter)**


In [ ]:
# Import required libraries
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output
import pandas as pd
import numpy as np
import json
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime, date

# Import from v2.0.0 package via compatibility layer
try:
    from compat_imports import (
        validate_comparative,
        export_to_json,
        export_to_markdown,
        export_to_csv,
        generate_radar_chart
    )
    utils_available = True
    print("✅ All imports successful from aimusic_eval package!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure the aimusic_eval package is installed: pip install -e .")
    utils_available = False

print("📊 Comparative Dashboard initialized")
print(f"🔧 Utils available: {utils_available}")


In [ ]:
# Initialize comparison data structure
comparison_data = {
    "template_metadata": {
        "template_type": "comparative_dashboard",
        "template_version": "1.0.0",
        "created_at": datetime.now().isoformat(),
        "modified_at": datetime.now().isoformat()
    },
    "comparison_metadata": {
        "evaluator": "",
        "num_systems": 0,
        "comparison_date": str(date.today())
    },
    "systems_compared": [],
    "score_matrix": {
        "criteria_scores": {},
        "average_scores": [],
        "statistics": {}
    },
    "criterion_comparisons": [],
    "patterns": {},
    "use_case_recommendations": {},
    "evaluator_synthesis": {},
    "visualizations": {}
}

# Storage for loaded synthesis data
loaded_syntheses = {}

print("✅ Data structure initialized")


---

## 1️⃣ Comparison Metadata

Define who is conducting this comparison and why.


In [ ]:
# Comparison metadata widgets
evaluator_widget = widgets.Text(
    description='Evaluator:',
    placeholder='Your name or ID',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

comparison_date_widget = widgets.DatePicker(
    description='Comparison Date:',
    value=date.today(),
    style={'description_width': '120px'}
)

comparison_purpose_widget = widgets.Textarea(
    description='Purpose:',
    placeholder='Why are you comparing these systems? (e.g., "System selection for production", "Research study", etc.)',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px', height='80px')
)

display(Markdown("### Enter comparison details:"))
display(evaluator_widget, comparison_date_widget, comparison_purpose_widget)


---

## 2️⃣ Import Synthesis Journals

Load synthesis JSON files from Phase 2 evaluations.

**Instructions:**
1. Enter paths to `synthesis_*.json` files (one per line)
2. Paths can be absolute or relative to this notebook
3. Click "📥 Load Syntheses" to import
4. Review the summary table to verify correct loading


In [ ]:
# Synthesis file import widgets
synthesis_paths_widget = widgets.Textarea(
    description='File Paths:',
    placeholder='Enter paths (one per line):\n../outputs/synthesis/synthesis_musicgen.json\n../outputs/synthesis/synthesis_ddsp.json\n...',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='700px', height='150px')
)

load_button = widgets.Button(
    description='📥 Load Syntheses',
    button_style='info',
    layout=widgets.Layout(width='200px')
)

import_output = widgets.Output()

def load_syntheses(b):
    """Load and parse synthesis JSON files"""
    with import_output:
        clear_output()

        # Get paths from textarea
        paths_text = synthesis_paths_widget.value.strip()
        if not paths_text:
            print("❌ No file paths entered")
            return

        paths = [p.strip() for p in paths_text.split('\n') if p.strip()]

        print(f"📂 Attempting to load {len(paths)} synthesis file(s)...")
        print("=" * 60)

        loaded_syntheses.clear()
        successful_loads = 0

        for i, path_str in enumerate(paths, 1):
            try:
                # Resolve path
                path = Path(path_str)
                if not path.is_absolute():
                    # Try relative to notebook location
                    path = Path.cwd() / path

                if not path.exists():
                    print(f"\n⚠️  File {i}: NOT FOUND")
                    print(f"   Path: {path}")
                    continue

                # Load JSON
                with open(path, 'r') as f:
                    data = json.load(f)

                # Extract system name
                system_name = data.get('system_metadata', {}).get('system_name', f'System_{i}')

                # Validate it's a synthesis journal
                if data.get('template_type') != 'synthesis_journal':
                    print(f"\n⚠️  File {i}: Not a synthesis journal")
                    print(f"   Type found: {data.get('template_type')}")
                    continue

                # Store loaded data
                loaded_syntheses[system_name] = {
                    'data': data,
                    'path': str(path),
                    'loaded_at': datetime.now().isoformat()
                }

                successful_loads += 1

                # Show summary
                num_criteria = len(data.get('quantitative_assessment', {}).get('criteria', []))
                num_sessions = len(data.get('imported_sessions', []))
                num_entries = len(data.get('reflective_journal', {}).get('entries', []))

                print(f"\n✅ Loaded: {system_name}")
                print(f"   📊 Criteria scored: {num_criteria}/8")
                print(f"   📝 Sessions imported: {num_sessions}")
                print(f"   📔 Journal entries: {num_entries}")

            except json.JSONDecodeError as e:
                print(f"\n❌ File {i}: Invalid JSON")
                print(f"   Error: {str(e)}")
            except Exception as e:
                print(f"\n❌ File {i}: Error loading")
                print(f"   Error: {str(e)}")

        print("\n" + "=" * 60)
        print(f"📊 SUMMARY: {successful_loads}/{len(paths)} files loaded successfully")

        if successful_loads >= 2:
            print(f"\n✅ Ready for comparison! ({successful_loads} systems loaded)")
            print("\n🔽 Scroll down to continue")
        elif successful_loads == 1:
            print("\n⚠️  Need at least 2 systems for comparison")
            print("   Please add more synthesis files")
        else:
            print("\n❌ No files loaded successfully")
            print("   Please check file paths and try again")

load_button.on_click(load_syntheses)

display(synthesis_paths_widget, load_button, import_output)


---

## 3️⃣ Loaded Systems Summary

Review the systems loaded for comparison.


In [ ]:
# Display loaded systems summary
summary_button = widgets.Button(
    description='📋 Show Summary',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

summary_output = widgets.Output()

def show_systems_summary(b):
    """Display summary table of loaded systems"""
    with summary_output:
        clear_output()

        if not loaded_syntheses:
            print("❌ No systems loaded yet")
            print("   Load synthesis files in the section above")
            return

        print("=" * 80)
        print("LOADED SYSTEMS SUMMARY")
        print("=" * 80)
        print(f"\nTotal Systems: {len(loaded_syntheses)}")
        print(f"Evaluator: {evaluator_widget.value or '(not specified)')}")
        print(f"Comparison Date: {comparison_date_widget.value}")
        print("\n" + "-" * 80)

        # Create summary data for table
        summary_data = []

        for system_name, synthesis_info in loaded_syntheses.items():
            data = synthesis_info['data']

            # Extract info
            system_version = data.get('system_metadata', {}).get('system_version', 'N/A')
            num_criteria = len(data.get('quantitative_assessment', {}).get('criteria', []))
            avg_score = data.get('quantitative_assessment', {}).get('average_score', 0)
            num_sessions = len(data.get('imported_sessions', []))
            num_entries = len(data.get('reflective_journal', {}).get('entries', []))

            summary_data.append({
                'System': system_name,
                'Version': system_version,
                'Avg Score': f"{avg_score:.2f}/5.00",
                'Criteria': f"{num_criteria}/8",
                'Sessions': num_sessions,
                'Entries': num_entries
            })

        # Display as DataFrame
        df = pd.DataFrame(summary_data)
        display(HTML(df.to_html(index=False, border=1)))

        print("\n✅ All systems loaded successfully")
        print("🔽 Scroll down to build score matrix and visualizations")

summary_button.on_click(show_systems_summary)

display(summary_button, summary_output)


---

## 4️⃣ Generate Score Matrix

Automatically extract scores from loaded syntheses and build comparison matrix.


In [ ]:
# Extract scores and build matrix
matrix_button = widgets.Button(
    description='📊 Generate Matrix',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

matrix_output = widgets.Output()

# Criteria mapping
CRITERIA_NAMES = [
    "Usability",
    "Generation Speed",
    "Audio Quality",
    "Stylistic Accuracy",
    "Parameter Control",
    "Content Generation Control",
    "DAW Integration",
    "Creative Workflow Fit"
]

def generate_score_matrix(b):
    """Extract scores from syntheses and build comparison matrix"""
    with matrix_output:
        clear_output()

        if not loaded_syntheses:
            print("❌ No systems loaded - load syntheses first")
            return

        if len(loaded_syntheses) < 2:
            print("⚠️  Need at least 2 systems for comparison")
            return

        print("📊 Generating score matrix...")
        print("=" * 80)

        # Extract system names in order
        system_names = list(loaded_syntheses.keys())
        num_systems = len(system_names)

        # Build score matrix
        matrix_data = []
        all_scores = {criterion: [] for criterion in CRITERIA_NAMES}

        for system_name in system_names:
            data = loaded_syntheses[system_name]['data']
            criteria = data.get('quantitative_assessment', {}).get('criteria', [])

            # Build dict for easy lookup
            criteria_dict = {c['criterion_name']: c for c in criteria}

            # Extract scores in order
            scores_row = []
            for criterion_name in CRITERIA_NAMES:
                if criterion_name in criteria_dict:
                    score = criteria_dict[criterion_name].get('score', 0)
                else:
                    score = 0  # Missing criterion

                scores_row.append(score)
                all_scores[criterion_name].append(score)

            matrix_data.append(scores_row)

        # Create DataFrame
        df = pd.DataFrame(matrix_data, index=system_names, columns=CRITERIA_NAMES)

        # Add average column
        df['AVERAGE'] = df.mean(axis=1).round(2)

        # Style the table with color coding
        def color_score(val):
            if val >= 4.0:
                return 'background-color: #90EE90'  # Light green
            elif val >= 3.0:
                return 'background-color: #FFFFE0'  # Light yellow
            elif val > 0:
                return 'background-color: #FFB6C1'  # Light red
            else:
                return 'background-color: #E0E0E0'  # Gray for N/A

        styled_df = df.style.applymap(color_score)

        print("\n📊 SCORE MATRIX")
        print("-" * 80)
        display(styled_df)

        # Calculate and display statistics
        print("\n" + "=" * 80)
        print("📈 STATISTICAL SUMMARY")
        print("=" * 80)

        stats_data = []
        for criterion in CRITERIA_NAMES:
            scores = all_scores[criterion]
            non_zero = [s for s in scores if s > 0]

            if non_zero:
                mean = np.mean(non_zero)
                std = np.std(non_zero)
                min_score = min(non_zero)
                max_score = max(non_zero)
                range_score = max_score - min_score

                # Find winner(s)
                max_idx = [i for i, s in enumerate(scores) if s == max_score]
                winners = [system_names[i] for i in max_idx]

                stats_data.append({
                    'Criterion': criterion,
                    'Mean': f"{mean:.2f}",
                    'Std Dev': f"{std:.2f}",
                    'Range': f"{range_score:.1f}",
                    'Winner(s)': ', '.join(winners)
                })

        stats_df = pd.DataFrame(stats_data)
        display(HTML(stats_df.to_html(index=False, border=1)))

        # Update comparison_data
        comparison_data['comparison_metadata']['num_systems'] = num_systems
        comparison_data['comparison_metadata']['evaluator'] = evaluator_widget.value
        comparison_data['systems_compared'] = [
            {
                'system_name': name,
                'synthesis_file': loaded_syntheses[name]['path']
            }
            for name in system_names
        ]
        comparison_data['score_matrix']['criteria_scores'] = all_scores
        comparison_data['score_matrix']['average_scores'] = df['AVERAGE'].tolist()

        print("\n✅ Score matrix generated successfully!")
        print("🔽 Scroll down to generate visualizations")

matrix_button.on_click(generate_score_matrix)

display(matrix_button, matrix_output)


---

## 5️⃣ Comparative Visualizations

Generate interactive charts for visual comparison.


### 5.1 Radar Chart (Overlay)

All systems on one 8-axis radar chart for direct visual comparison.


In [ ]:
# Generate overlay radar chart
radar_button = widgets.Button(
    description='📈 Generate Radar Chart',
    button_style='info',
    layout=widgets.Layout(width='250px')
)

radar_output = widgets.Output()

def generate_radar_comparison(b):
    """Generate overlay radar chart with all systems"""
    with radar_output:
        clear_output()

        if not loaded_syntheses:
            print("❌ No systems loaded")
            return

        system_names = list(loaded_syntheses.keys())

        # Extract scores for each system
        fig = go.Figure()

        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

        for i, system_name in enumerate(system_names):
            data = loaded_syntheses[system_name]['data']
            criteria = data.get('quantitative_assessment', {}).get('criteria', [])
            criteria_dict = {c['criterion_name']: c for c in criteria}

            scores = []
            for criterion_name in CRITERIA_NAMES:
                score = criteria_dict.get(criterion_name, {}).get('score', 0)
                scores.append(score)

            # Close the polygon
            scores_closed = scores + [scores[0]]
            labels_closed = CRITERIA_NAMES + [CRITERIA_NAMES[0]]

            fig.add_trace(go.Scatterpolar(
                r=scores_closed,
                theta=labels_closed,
                fill='toself',
                name=system_name,
                line=dict(color=colors[i % len(colors)])
            ))

        fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 5]
                )
            ),
            showlegend=True,
            title="Multi-System Performance Comparison",
            height=600
        )

        display(fig)

        comparison_data['visualizations']['radar_chart_generated'] = True
        print("\n✅ Radar chart generated")

radar_button.on_click(generate_radar_comparison)

display(radar_button, radar_output)


### 5.2 Heatmap

Color-coded score matrix for quick pattern identification.


In [ ]:
# Generate heatmap
heatmap_button = widgets.Button(
    description='🔥 Generate Heatmap',
    button_style='info',
    layout=widgets.Layout(width='250px')
)

heatmap_output = widgets.Output()

def generate_heatmap(b):
    """Generate heatmap visualization"""
    with heatmap_output:
        clear_output()

        if not loaded_syntheses:
            print("❌ No systems loaded")
            return

        system_names = list(loaded_syntheses.keys())

        # Build matrix
        matrix = []
        for system_name in system_names:
            data = loaded_syntheses[system_name]['data']
            criteria = data.get('quantitative_assessment', {}).get('criteria', [])
            criteria_dict = {c['criterion_name']: c for c in criteria}

            scores = [criteria_dict.get(cn, {}).get('score', 0) for cn in CRITERIA_NAMES]
            matrix.append(scores)

        # Create heatmap
        fig = go.Figure(data=go.Heatmap(
            z=matrix,
            x=CRITERIA_NAMES,
            y=system_names,
            colorscale='RdYlGn',
            zmin=0,
            zmax=5,
            text=matrix,
            texttemplate="%{text:.1f}",
            textfont={"size": 10},
            colorbar=dict(title="Score")
        ))

        fig.update_layout(
            title="Performance Heatmap",
            xaxis_title="Performance Criteria",
            yaxis_title="AI Music Systems",
            height=400,
            xaxis=dict(tickangle=-45)
        )

        display(fig)

        comparison_data['visualizations']['heatmap_generated'] = True
        print("\n✅ Heatmap generated")

heatmap_button.on_click(generate_heatmap)

display(heatmap_button, heatmap_output)


### 5.3 Grouped Bar Chart

Side-by-side comparison per criterion.


In [ ]:
# Generate grouped bar chart
bar_button = widgets.Button(
    description='📊 Generate Bar Chart',
    button_style='info',
    layout=widgets.Layout(width='250px')
)

bar_output = widgets.Output()

def generate_bar_chart(b):
    """Generate grouped bar chart"""
    with bar_output:
        clear_output()

        if not loaded_syntheses:
            print("❌ No systems loaded")
            return

        system_names = list(loaded_syntheses.keys())

        fig = go.Figure()

        for system_name in system_names:
            data = loaded_syntheses[system_name]['data']
            criteria = data.get('quantitative_assessment', {}).get('criteria', [])
            criteria_dict = {c['criterion_name']: c for c in criteria}

            scores = [criteria_dict.get(cn, {}).get('score', 0) for cn in CRITERIA_NAMES]

            fig.add_trace(go.Bar(
                name=system_name,
                x=CRITERIA_NAMES,
                y=scores,
                text=scores,
                textposition='outside'
            ))

        fig.update_layout(
            barmode='group',
            title="Criterion-by-Criterion Comparison",
            xaxis_title="Performance Criteria",
            yaxis_title="Score (0-5)",
            yaxis=dict(range=[0, 5.5]),
            height=500,
            xaxis=dict(tickangle=-45)
        )

        display(fig)

        comparison_data['visualizations']['bar_chart_generated'] = True
        print("\n✅ Bar chart generated")

bar_button.on_click(generate_bar_chart)

display(bar_button, bar_output)


---

## 6️⃣ Criterion-by-Criterion Analysis

Detailed qualitative comparison for each of the 8 criteria.

For each criterion below:
- Winner is auto-identified from scores
- Evidence is auto-extracted from synthesis journals
- Provide your comparative analysis in the textareas


In [ ]:
# Criterion comparison widgets storage
criterion_widgets = {}

def create_criterion_section(criterion_name):
    """Create comparison section for one criterion"""

    # Auto-extract winner and evidence
    system_names = list(loaded_syntheses.keys())

    if system_names:
        # Get scores
        scores = []
        for system_name in system_names:
            data = loaded_syntheses[system_name]['data']
            criteria = data.get('quantitative_assessment', {}).get('criteria', [])
            criteria_dict = {c['criterion_name']: c for c in criteria}
            score = criteria_dict.get(criterion_name, {}).get('score', 0)
            scores.append((system_name, score))

        # Find winner
        max_score = max(s[1] for s in scores)
        winners = [s[0] for s in scores if s[1] == max_score]
        winner_text = ', '.join(winners) + f" ({max_score}/5)"
    else:
        winner_text = "(Load systems first)"

    # Widgets
    winner_display = widgets.HTML(f"<b>Winner:</b> {winner_text}")

    performance_comparison = widgets.Textarea(
        description='Performance Comparison:',
        placeholder='Describe how systems differ on this criterion...',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='700px', height='100px')
    )

    root_cause = widgets.Textarea(
        description='Root Cause Analysis:',
        placeholder='Why do these differences exist? (architecture, design, model, etc.)',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='700px', height='100px')
    )

    practical_implications = widgets.Textarea(
        description='Practical Implications:',
        placeholder='How do these differences affect real-world use?',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='700px', height='100px')
    )

    criterion_widgets[criterion_name] = {
        'winner_display': winner_display,
        'performance_comparison': performance_comparison,
        'root_cause': root_cause,
        'practical_implications': practical_implications
    }

    return widgets.VBox([
        winner_display,
        performance_comparison,
        root_cause,
        practical_implications
    ])

print("✅ Criterion comparison widgets ready")
print("   Run the next cells to display each criterion")


### 6.1 Usability

In [ ]:
display(create_criterion_section("Usability"))

### 6.2 Generation Speed

In [ ]:
display(create_criterion_section("Generation Speed"))

### 6.3 Audio Quality

In [ ]:
display(create_criterion_section("Audio Quality"))

### 6.4 Stylistic Accuracy

In [ ]:
display(create_criterion_section("Stylistic Accuracy"))

### 6.5 Parameter Control

In [ ]:
display(create_criterion_section("Parameter Control"))

### 6.6 Content Generation Control

In [ ]:
display(create_criterion_section("Content Generation Control"))

### 6.7 DAW Integration

In [ ]:
display(create_criterion_section("DAW Integration"))

### 6.8 Creative Workflow Fit

In [ ]:
display(create_criterion_section("Creative Workflow Fit"))

---

## 7️⃣ Pattern Identification

Identify universal patterns and distinctive capabilities across systems.


In [ ]:
# Universal patterns
universal_strengths = widgets.Textarea(
    description='Universal Strengths:',
    placeholder='List strengths ALL systems share (one per line):\n- Strength 1\n- Strength 2\n- ...',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='100px')
)

universal_limitations = widgets.Textarea(
    description='Universal Limitations:',
    placeholder='List limitations ALL systems share (one per line):\n- Limitation 1\n- Limitation 2\n- ...',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='100px')
)

field_insights = widgets.Textarea(
    description='Field-Level Insights:',
    placeholder='What do these patterns reveal about the current state of AI music generation?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='120px')
)

display(Markdown("### Universal Patterns"))
display(universal_strengths, universal_limitations, field_insights)


In [ ]:
# Distinctive capabilities per system
distinctive_capabilities = widgets.Textarea(
    description='Distinctive Capabilities:',
    placeholder='For each system, what makes it UNIQUE?\n\nSystem A:\n- Unique strength 1\n- Unique strength 2\n\nSystem B:\n- ...',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='150px')
)

display(Markdown("### Distinctive Capabilities"))
display(distinctive_capabilities)


---

## 8️⃣ Use Case Recommendations

Map which system works best for specific tasks and user profiles.


In [ ]:
# Task-based recommendations
use_case_tasks = [
    'Quick Ideation',
    'Composition',
    'Arrangement',
    'Sound Design',
    'Specific Instrument Generation',
    'Rhythmic Elements',
    'Melodic/Harmonic Elements',
    'Textures/Ambience',
    'Final Production'
]

task_recommendations = {}

system_names = list(loaded_syntheses.keys()) if loaded_syntheses else ['System 1', 'System 2']

display(Markdown("### Task-Based Recommendations"))
display(Markdown("*Select the best system for each production task:*"))

for task in use_case_tasks:
    dropdown = widgets.Dropdown(
        options=system_names,
        description=f'{task}:',
        style={'description_width': '200px'},
        layout=widgets.Layout(width='450px')
    )
    task_recommendations[task] = dropdown
    display(dropdown)


In [ ]:
# User profile recommendations
display(Markdown("### User Profile Recommendations"))

profile_beginners = widgets.Dropdown(
    options=system_names,
    description='Best for Beginners:',
    style={'description_width': '150px'}
)

profile_beginners_rationale = widgets.Textarea(
    description='Why:',
    placeholder='Why is this system best for beginners?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='60px')
)

profile_professionals = widgets.Dropdown(
    options=system_names,
    description='Best for Professionals:',
    style={'description_width': '150px'}
)

profile_professionals_rationale = widgets.Textarea(
    description='Why:',
    placeholder='Why is this system best for professionals?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='60px')
)

profile_experimental = widgets.Dropdown(
    options=system_names,
    description='Best for Experimental:',
    style={'description_width': '150px'}
)

profile_experimental_rationale = widgets.Textarea(
    description='Why:',
    placeholder='Why is this system best for experimental artists?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='60px')
)

display(profile_beginners, profile_beginners_rationale)
display(profile_professionals, profile_professionals_rationale)
display(profile_experimental, profile_experimental_rationale)


---

## 9️⃣ Evaluator Synthesis

Your overall insights and final recommendations.


In [ ]:
# Evaluator synthesis
synthesis_surprises = widgets.Textarea(
    description='Surprises:',
    placeholder='What unexpected patterns or differences emerged from this comparison?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='100px')
)

synthesis_confirmations = widgets.Textarea(
    description='Confirmations:',
    placeholder='What hypotheses or expectations were validated?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='100px')
)

synthesis_questions = widgets.Textarea(
    description='Open Questions:',
    placeholder='What remains unclear or requires further investigation?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='100px')
)

evaluator_positionality = widgets.Textarea(
    description='Evaluator Positionality:',
    placeholder='How did your expertise, background, or perspective influence these comparative judgments?',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='100px')
)

display(Markdown("### Synthesis & Reflections"))
display(synthesis_surprises, synthesis_confirmations, synthesis_questions, evaluator_positionality)


In [ ]:
# Final recommendation
display(Markdown("### Final Recommendation"))
display(Markdown("*If you could only use ONE system:*"))

final_choice = widgets.Dropdown(
    options=system_names,
    description='Single Choice:',
    style={'description_width': '150px'}
)

final_rationale = widgets.Textarea(
    description='Rationale:',
    placeholder='Why this system above all others? Synthesize all quantitative and qualitative insights.',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='700px', height='120px')
)

display(final_choice, final_rationale)


---

## 🔟 Export Comparative Analysis

Save your comparison in multiple formats for analysis and reporting.


In [ ]:
# Collect all comparison data
def collect_comparison_data():
    """Gather all entered data for export"""

    # Update metadata
    comparison_data['comparison_metadata']['evaluator'] = evaluator_widget.value
    comparison_data['comparison_metadata']['comparison_date'] = str(comparison_date_widget.value)
    comparison_data['comparison_metadata']['comparison_purpose'] = comparison_purpose_widget.value

    # Criterion comparisons
    criterion_comparisons = []
    for criterion_name in CRITERIA_NAMES:
        if criterion_name in criterion_widgets:
            widgets = criterion_widgets[criterion_name]
            criterion_comparisons.append({
                'criterion_name': criterion_name,
                'performance_comparison': widgets['performance_comparison'].value,
                'root_cause_analysis': widgets['root_cause'].value,
                'practical_implications': widgets['practical_implications'].value
            })

    comparison_data['criterion_comparisons'] = criterion_comparisons

    # Patterns
    comparison_data['patterns'] = {
        'universal_strengths': [s.strip() for s in universal_strengths.value.split('\n') if s.strip()],
        'universal_limitations': [s.strip() for s in universal_limitations.value.split('\n') if s.strip()],
        'field_level_insights': field_insights.value,
        'distinctive_capabilities': distinctive_capabilities.value
    }

    # Use case recommendations
    comparison_data['use_case_recommendations'] = {
        'task_mappings': {task: task_recommendations[task].value for task in use_case_tasks},
        'user_profiles': {
            'beginners': {
                'recommended_system': profile_beginners.value,
                'rationale': profile_beginners_rationale.value
            },
            'professionals': {
                'recommended_system': profile_professionals.value,
                'rationale': profile_professionals_rationale.value
            },
            'experimental_artists': {
                'recommended_system': profile_experimental.value,
                'rationale': profile_experimental_rationale.value
            }
        }
    }

    # Evaluator synthesis
    comparison_data['evaluator_synthesis'] = {
        'surprises': synthesis_surprises.value,
        'confirmations': synthesis_confirmations.value,
        'open_questions': synthesis_questions.value,
        'evaluator_positionality': evaluator_positionality.value,
        'final_recommendation': {
            'single_choice': final_choice.value,
            'rationale': final_rationale.value
        }
    }

    # Update timestamp
    comparison_data['template_metadata']['modified_at'] = datetime.now().isoformat()

    return comparison_data

print("✅ Data collection function ready")


In [ ]:
# Export functionality
export_button = widgets.Button(
    description='💾 Export JSON',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

export_output_widget = widgets.Output()

def export_comparison(b):
    """Export comparison to JSON file"""
    with export_output_widget:
        clear_output()

        data = collect_comparison_data()

        # Create outputs directory
        output_dir = Path('../outputs/comparisons')
        output_dir.mkdir(parents=True, exist_ok=True)

        # Generate filename
        num_systems = len(loaded_syntheses)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"comparative_analysis_{num_systems}systems_{timestamp}.json"
        filepath = output_dir / filename

        try:
            # Add export timestamp
            data['template_metadata']['exported_at'] = datetime.now().isoformat()

            # Export JSON
            with open(filepath, 'w') as f:
                json.dump(data, f, indent=2)

            print("=" * 70)
            print("EXPORT SUCCESSFUL")
            print("=" * 70)
            print(f"\n✅ JSON exported: {filepath}")
            print(f"   File size: {filepath.stat().st_size} bytes")
            print(f"   Systems compared: {num_systems}")

            # Also export markdown report
            md_filename = filename.replace('.json', '.md')
            md_filepath = output_dir / md_filename

            # Generate markdown report
            with open(md_filepath, 'w') as f:
                f.write(f"# Comparative Analysis Report\n\n")
                f.write(f"**Date:** {data['comparison_metadata']['comparison_date']}\n")
                f.write(f"**Evaluator:** {data['comparison_metadata']['evaluator']}\n")
                f.write(f"**Systems Compared:** {num_systems}\n\n")

                f.write(f"## Systems\n\n")
                for sys_info in data['systems_compared']:
                    f.write(f"- **{sys_info['system_name']}**\n")

                f.write(f"\n## Score Matrix\n\n")
                # Add score table

                f.write(f"\n## Analysis\n\n")
                f.write(f"{data['evaluator_synthesis'].get('final_recommendation', {}).get('rationale', '')}\n")

            print(f"\n✅ Markdown report exported: {md_filepath}")

            print(f"\n📁 Location: {output_dir.absolute()}")
            print(f"\n🎉 Export complete!")

        except Exception as e:
            print(f"❌ Export error: {str(e)}")

export_button.on_click(export_comparison)

display(export_button, export_output_widget)


---

## 📊 Optional: Advanced Statistical Analysis (Phase 4)

### For Power Users and Publication Preparation

After exporting your comparative analysis, you can optionally use the **Automated Comparison Utility** for advanced statistical analysis and publication-ready outputs.

#### When to Use the Automated Utility

✅ **Use when you need:**
- **Statistical rigor** - Effect sizes (Cohen's d), correlation analysis, clustering
- **5+ systems** - Batch processing for large-scale comparisons
- **Publication outputs** - High-resolution visualizations (300 DPI), LaTeX tables
- **Deep insights** - Criterion relationships, performance groupings, significance testing
- **Supplementary materials** - Auto-generated statistical reports for papers

❌ **Skip if:**
- You're comparing **2-4 systems** (this dashboard provides sufficient analysis)
- **Exploratory analysis** is your goal (not formal publication)
- **Quick comparisons** are all you need
- You're in **early research phases** (not ready for statistical rigor)

#### What the Utility Provides

The **Automated Comparison Utility** (`automated_comparison_utility.ipynb`) analyzes your exported `comparative_analysis_*.json` file and generates:

1. **Advanced Statistics:**
   - Descriptive statistics (mean, std dev, range, IQR) per criterion and per system
   - Effect size calculations (Cohen's d) for all pairwise system comparisons
   - Pearson correlation matrix between criteria (reveals criterion relationships)
   - Hierarchical clustering analysis (groups systems by performance profiles)

2. **Publication-Ready Visualizations (300 DPI):**
   - Enhanced radar charts
   - Grouped bar charts
   - Score heatmaps
   - Correlation matrices
   - Dendrograms (clustering trees)

3. **Exports for Academic Papers:**
   - High-resolution PNG images (suitable for journals)
   - LaTeX tables (copy-paste into manuscripts)
   - Markdown supplementary materials documents
   - Suggested methods section text for papers

#### How to Use It

1. **Complete this dashboard** and export your comparative analysis (JSON format)
2. **Open** [`automated_comparison_utility.ipynb`](automated_comparison_utility.ipynb)
3. **Run all cells** - The utility will automatically find and load your export
4. **Review** the statistical analyses and visualizations
5. **Export** publication materials to `../outputs/visualizations/` and `../outputs/supplementary/`

#### Important Statistical Note

Traditional statistical tests (ANOVA, t-tests) require **multiple replications** per system (e.g., 3+ independent evaluations). Since the framework typically involves **single synthesized scores per system**, the utility focuses on:

- **Descriptive statistics** to characterize performance
- **Effect sizes** (Cohen's d) to measure practical significance
- **Correlation patterns** to understand criterion relationships
- **Clustering methods** to reveal natural system groupings

These approaches provide valuable insights without requiring the assumptions of traditional hypothesis testing.

#### Quick Reference

| Task | Use This Dashboard | Use Automated Utility |
|------|-------------------|----------------------|
| Compare 2-4 systems | ✅ Perfect | ❌ Overkill |
| Compare 5+ systems | ✅ Good | ✅ Better |
| Publication preparation | ✅ Sufficient | ✅ Recommended |
| Statistical significance | ❌ Not available | ⚠️ Limited (see note above) |
| LaTeX tables | ❌ No | ✅ Yes |
| High-res visualizations | ❌ No | ✅ Yes (300 DPI) |
| Correlation analysis | ❌ No | ✅ Yes |
| Clustering analysis | ❌ No | ✅ Yes |

---

**Next Notebook:** [automated_comparison_utility.ipynb](automated_comparison_utility.ipynb) *(optional - for advanced analysis)*

---


---

## ✅ Comparative Analysis Complete!

### What You've Accomplished

You've completed a comprehensive multi-system comparison including:

- ✅ Imported 2+ synthesis journals from Phase 2
- ✅ Generated score matrix with statistical analysis
- ✅ Created comparative visualizations (radar, heatmap, bar charts)
- ✅ Analyzed all 8 criteria in detail
- ✅ Identified universal patterns and distinctive capabilities
- ✅ Mapped use cases and user profiles
- ✅ Provided final synthesis and recommendations
- ✅ Exported complete comparison data

### Output Files

Your comparison has been saved to:
- `../outputs/comparisons/comparative_analysis_*.json` - Complete structured data
- `../outputs/comparisons/comparative_analysis_*.md` - Human-readable report

### Next Steps

1. **Review exported files** - Check your JSON and Markdown reports
2. **Share with stakeholders** - Use for decision-making or publication
3. **Continue evaluation** - Add more systems and re-compare
4. **Create new session notebooks** - Document additional experiences with systems

### Related Templates

- **The Session Notebook** - Capture individual evaluation sessions
- **The Synthesis Journal** - Assess and reflect on single systems
- **Automated Comparison Utility** (Phase 4) - Statistical analysis tool

---

**📚 Framework Documentation:** See project README for complete guide

**Phase 3 of 7→3+1 Template Consolidation**
**Replaces:** Template 04 (Comparative Analysis)
